# Lab 2 - Kaggle Titanic Challenge

## 1. Imports & Classifier Architecture

In [17]:
import random
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

def calc_error_rate(actual, predicted):
    actual = list(actual)
    predicted = list(predicted)
    wrong = sum(1 for a, p in zip(actual, predicted) if a != p)
    return wrong / len(actual)

class BaseClassifier:
    def fit(self, X, y):
        pass

    def predict(self, X):
        pass

    def score(self, X, y):
        preds = self.predict(X)
        return sum(1 for a, p in zip(list(y), preds) if a == p) / len(y)

    def error(self, X, y):
        return 1.0 - self.score(X, y)

class MajorityVoteClassifier(BaseClassifier):
    def __init__(self):
        self.mode_label = None

    def fit(self, X, y):
        y_list = list(y)
        self.mode_label = max(set(y_list), key=y_list.count)
        return self

    def predict(self, X):
        n = len(X)
        return [self.mode_label] * n

class MemorizerClassifier(BaseClassifier):
    def __init__(self):
        self.memory_bank = {}
        self.all_classes = []

    def fit(self, X, y):
        self.memory_bank.clear()
        self.all_classes = list(set(y))
        matrix = X.to_numpy() if hasattr(X, "to_numpy") else X
        for row, label in zip(matrix, y):
            self.memory_bank[tuple(row)] = label
        return self

    def predict(self, X):
        matrix = X.to_numpy() if hasattr(X, "to_numpy") else X
        outputs = []
        for row in matrix:
            t_row = tuple(row)
            if t_row in self.memory_bank:
                outputs.append(self.memory_bank[t_row])
            else:
                outputs.append(random.choice(self.all_classes))
        return outputs

class DecisionStumpClassifier(BaseClassifier):
    def __init__(self, feature_col=0):
        self.feature_col = feature_col
        self.rules_dict = {}
        self.overall_mode = None

    def fit(self, X, y):
        y_list = list(y)
        matrix = X.to_numpy() if hasattr(X, "to_numpy") else X
        
        splits = {}
        for row, label in zip(matrix, y_list):
            val = row[self.feature_col]
            splits.setdefault(val, []).append(label)
        
        self.rules_dict = {
            val: max(set(labels), key=labels.count)
            for val, labels in splits.items()
        }
        self.overall_mode = max(set(y_list), key=y_list.count)
        return self

    def predict(self, X):
        matrix = X.to_numpy() if hasattr(X, "to_numpy") else X
        return [
            self.rules_dict.get(row[self.feature_col], self.overall_mode)
            for row in matrix
        ]

## Step 1 & 2: Load and Prepare Titanic Dataset

In [18]:
raw_train = pd.read_csv("train.csv")
raw_test = pd.read_csv("test.csv")

print("Train dimension:", raw_train.shape)
print("Test dimension:", raw_test.shape)
print("\nSurvival breakdown:")
print(raw_train["Survived"].value_counts())

feature_cols = ["Pclass", "Sex", "Age", "Fare"]

# Preprocessing pipeline
med_age = raw_train["Age"].median()
med_fare = raw_train["Fare"].median()

def clean_features(df):
    data = df[feature_cols].copy()
    # Convert Sex to binary: female -> 1, male -> 0
    data["Sex"] = (data["Sex"] == "female").astype(int)
    data["Age"] = data["Age"].fillna(med_age)
    data["Fare"] = data["Fare"].fillna(med_fare)
    return data

X = clean_features(raw_train)
y = raw_train["Survived"]
X_test = clean_features(raw_test)

# 80/20 train-validation split (random_state=42 as per lab manual)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTraining set: {X_train.shape} | Validation set: {X_val.shape}")

Train dimension: (891, 12)
Test dimension: (418, 11)

Survival breakdown:
Survived
0    549
1    342
Name: count, dtype: int64

Training set: (712, 4) | Validation set: (179, 4)


## Step 3: Majority Vote Baseline

In [19]:
mv_model = MajorityVoteClassifier()
mv_model.fit(X_train, y_train)

mv_tr_acc = 1 - calc_error_rate(y_train, mv_model.predict(X_train))
mv_val_acc = 1 - calc_error_rate(y_val, mv_model.predict(X_val))

print(f"Majority Vote -> Train Acc: {mv_tr_acc:.4f}, Val Acc: {mv_val_acc:.4f}")

Majority Vote -> Train Acc: 0.6166, Val Acc: 0.6145


## Step 4: Memorizer Classifier

In [20]:
mem_model = MemorizerClassifier()
mem_model.fit(X_train, y_train)

mem_tr_acc = 1 - calc_error_rate(y_train, mem_model.predict(X_train))
mem_val_acc = 1 - calc_error_rate(y_val, mem_model.predict(X_val))

print(f"Memorizer -> Train Acc: {mem_tr_acc:.4f}, Val Acc: {mem_val_acc:.4f}")

Memorizer -> Train Acc: 0.9761, Val Acc: 0.6927


## Step 5: Decision Stumps (Single Features)

In [21]:
stump_performances = {}

for col_idx, col_name in enumerate(feature_cols):
    stump = DecisionStumpClassifier(feature_col=col_idx)
    stump.fit(X_train, y_train)
    tr_acc = 1 - calc_error_rate(y_train, stump.predict(X_train))
    val_acc = 1 - calc_error_rate(y_val, stump.predict(X_val))
    stump_performances[col_name] = (stump, tr_acc, val_acc)
    print(f"Feature [{col_name:<6}] -> Train Acc: {tr_acc:.4f} | Val Acc: {val_acc:.4f}")

top_feature = max(stump_performances, key=lambda f: stump_performances[f][1])
best_stump_obj, stump_tr_acc, stump_val_acc = stump_performances[top_feature]
print(f"\nHighest performing individual feature: {top_feature}")

Feature [Pclass] -> Train Acc: 0.6882 | Val Acc: 0.6425
Feature [Sex   ] -> Train Acc: 0.7893 | Val Acc: 0.7765
Feature [Age   ] -> Train Acc: 0.6826 | Val Acc: 0.5810
Feature [Fare  ] -> Train Acc: 0.8258 | Val Acc: 0.6369

Highest performing individual feature: Fare


## Step 6: Decision Tree Model

In [22]:
dt_model = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_model.fit(X_train, y_train)

dt_tr_acc = (dt_model.predict(X_train) == y_train).mean()
dt_val_acc = (dt_model.predict(X_val) == y_val).mean()

print(f"Decision Tree (depth=3) -> Train Acc: {dt_tr_acc:.4f}, Val Acc: {dt_val_acc:.4f}")

Decision Tree (depth=3) -> Train Acc: 0.8258, Val Acc: 0.7933


## Step 7 & 8: Generate Kaggle Submission File

In [23]:
# Retrain final Decision Tree on the entire training set
final_classifier = DecisionTreeClassifier(max_depth=3, random_state=42)
final_classifier.fit(X, y)

test_preds = final_classifier.predict(X_test)

submission_df = pd.DataFrame({
    "PassengerId": raw_test["PassengerId"],
    "Survived": test_preds
})

submission_df.to_csv("submission.csv", index=False)
print(f"Saved submission.csv with shape: {submission_df.shape}")
print(submission_df.head())

Saved submission.csv with shape: (418, 2)
   PassengerId  Survived
0          892         0
1          893         1
2          894         0
3          895         0
4          896         1


## Step 9: Final Performance Comparison

In [24]:
print(f"{'Model':<28} | {'Training Acc':<14} | {'Validation Acc':<14}")
print("-" * 62)
print(f"{'Majority Vote':<28} | {mv_tr_acc:<14.3f} | {mv_val_acc:<14.3f}")
print(f"{'Memorizer':<28} | {mem_tr_acc:<14.3f} | {mem_val_acc:<14.3f}")
print(f"{'Decision Stump (' + top_feature + ')':<28} | {stump_tr_acc:<14.3f} | {stump_val_acc:<14.3f}")
print(f"{'Decision Tree (depth=3)':<28} | {dt_tr_acc:<14.3f} | {dt_val_acc:<14.3f}")

Model                        | Training Acc   | Validation Acc
--------------------------------------------------------------
Majority Vote                | 0.617          | 0.615         
Memorizer                    | 0.976          | 0.693         
Decision Stump (Fare)        | 0.826          | 0.637         
Decision Tree (depth=3)      | 0.826          | 0.793         
